# Customer Lifetime Value Model
- Justin Wall, Senior Data Scientist, Ridgeline Outfitters

---
## Questions to answer
1. Some way to estimate future value per customer (or per segment) — not just "what have they spent so far," but what we should expect over the next 12 months.
2. Confidence ranges, not just point estimates — I don't want to walk into a budget meeting with a number that turns out to be 3x off in practice.
3. A clear way to compare that future value against what we're paying to acquire or retain people, so we can actually set sane spend limits.

## Load Data

In [3]:
import pandas as pd

df = pd.read_csv('../data/ridgeline_outfitters_transactions.csv', parse_dates=['transaction_date'])

print(df.shape)
print('-------------------------------')
print(df.dtypes)
print('-------------------------------')
print(df.head())
print('-------------------------------')
print(df.describe())

(5282, 3)
-------------------------------
customer_id                    str
transaction_date    datetime64[us]
order_value                float64
dtype: object
-------------------------------
  customer_id transaction_date  order_value
0    RO-10000       2023-01-02       261.92
1    RO-10000       2023-04-08       103.40
2    RO-10000       2023-08-27        46.61
3    RO-10000       2023-11-14        37.09
4    RO-10000       2024-05-11        44.66
-------------------------------
                 transaction_date  order_value
count                        5282  5282.000000
mean   2023-08-11 04:36:59.159409   181.120015
min           2022-01-01 00:00:00    15.000000
25%           2023-01-05 00:00:00    67.685000
50%           2023-08-03 00:00:00   132.590000
75%           2024-03-30 00:00:00   244.315000
max           2024-12-31 00:00:00   800.000000
std                           NaN   158.015204


In [4]:
# Transaction date range
print(f"Date range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")

# Customer counts
print(f"Unique customers: {df['customer_id'].nunique()}")
print(f"Total transactions: {len(df)}")

# Order value
print(f"\nOrder value summary:")
print(df['order_value'].describe().round(2))

# Transactions per customer
txn_per_customer = df.groupby('customer_id').size()
print(f"\nTransactions per customer:")
print(txn_per_customer.describe().round(2))

Date range: 2022-01-01 to 2024-12-31
Unique customers: 1000
Total transactions: 5282

Order value summary:
count    5282.00
mean      181.12
std       158.02
min        15.00
25%        67.68
50%       132.59
75%       244.32
max       800.00
Name: order_value, dtype: float64

Transactions per customer:
count    1000.00
mean        5.28
std         6.80
min         1.00
25%         1.00
50%         3.00
75%         6.00
max        28.00
dtype: float64


## Data Alignment

In [5]:
from lifetimes.utils import summary_data_from_transaction_data

CALIBRATION_END = pd.Timestamp('2024-06-30')

calib_df = df[df['transaction_date'] <= CALIBRATION_END]

rfm = summary_data_from_transaction_data(
    calib_df,
    customer_id_col='customer_id',
    datetime_col='transaction_date',
    monetary_value_col='order_value',
    observation_period_end=CALIBRATION_END,
    freq='D'
)

print(rfm.shape)
print(rfm.head(10))
print(rfm.describe().round(2))

(1000, 4)
             frequency  recency      T  monetary_value
customer_id                                           
RO-10000           4.0    495.0  545.0       57.940000
RO-10001           0.0      0.0  583.0        0.000000
RO-10002          16.0    627.0  661.0      136.392500
RO-10003           1.0    145.0  425.0      627.980000
RO-10004           0.0      0.0  742.0        0.000000
RO-10005           3.0    191.0  376.0       15.613333
RO-10006           4.0    678.0  700.0      261.672500
RO-10007           3.0    330.0  545.0      486.346667
RO-10008           0.0      0.0  803.0        0.000000
RO-10009           5.0    688.0  696.0      544.314000
       frequency  recency        T  monetary_value
count    1000.00  1000.00  1000.00         1000.00
mean        3.41   261.63   631.71           95.27
std         5.35   285.52   156.64          119.79
min         0.00     0.00   372.00            0.00
25%         0.00     0.00   493.75            0.00
50%         1.00   188.5

In [6]:
print(f"Customers with only 1 purchase (frequency=0): {(rfm['frequency'] == 0).sum()}")
print(f"Customers with repeat purchases:              {(rfm['frequency'] > 0).sum()}")
print(f"Average repeat purchase monetary value:       ${rfm[rfm['frequency'] > 0]['monetary_value'].mean():.2f}")
print(f"Max customer age in dataset (days):           {rfm['T'].max():.0f}")

Customers with only 1 purchase (frequency=0): 462
Customers with repeat purchases:              538
Average repeat purchase monetary value:       $177.08
Max customer age in dataset (days):           911


## Data Analysis

## Modeling & Validation

## Insights